# All-rundle pairwise agreement

Interactive front-end for `ll_all_rundle_analysis.py`. The expensive step (`build_agreement_tables`) runs once in this kernel session; every cell after that is a cheap lookup against the precomputed tables, so you can re-run queries against different rundles/players without rebuilding anything.

In [1]:
import json
import os
import time

from ll_all_rundle_analysis import (
    build_agreement_tables,
    pairwise_agreement,
    one_pairwise_agreement,
    average_agreement,
    most_similar_pairs,
    least_similar_pairs,
    agreement_dict_for_rundle,
)
from ll_analysis import print_agreement_matrix

In [2]:
DATA_FILE = "rundle_data.json"

with open(DATA_FILE) as f:
    data = json.load(f)

print(f"season {data['season']}: {data['num_days']} days x {data['num_questions']} questions/day")
print(f"{len(data['rundles'])} rundle entries ({sum(1 for r in data['rundles'].values() if r)} non-empty)")

season 108: 25 days x 6 questions/day
1424 rundle entries (1291 non-empty)


In [3]:
t0 = time.time()
tables = build_agreement_tables(data)
print(f"built tables for {len(tables)} (rundle, player) rows in {time.time() - t0:.1f}s")

built tables for 35858 (rundle, player) rows in 16.2s


## Pick a rundle to explore

Rundle names look like `A_Aloha`, `B_Beach`, etc. -- the letter prefix before the underscore is the tier/branch (`A` through `E`, plus `R` for Rookie).

In [ ]:
sorted(tables.index_by_rundle.keys())[:20]

In [ ]:
RUNDLE = sorted(tables.index_by_rundle.keys())[0]
print(f"using rundle: {RUNDLE}")
print_agreement_matrix(agreement_dict_for_rundle(tables, RUNDLE))

In [ ]:
print(f"Most similar pairs in {RUNDLE}:")
for (a, b), val in most_similar_pairs(tables, n=10, rundle=RUNDLE):
    print(f"  {a[1]} & {b[1]}: {val:.1%}")

print(f"\nLeast similar pairs in {RUNDLE}:")
for (a, b), val in least_similar_pairs(tables, n=10, rundle=RUNDLE):
    print(f"  {a[1]} & {b[1]}: {val:.1%}")

In [ ]:
print(f"Average agreement within {RUNDLE}:")
for label, avg in average_agreement(tables, rundle=RUNDLE):
    print(f"  {label[1]:<14} {avg:.1%}" if avg is not None else f"  {label[1]} n/a")

## Single player / single pair lookups

Names are usually unique, so plain `"LastFirst"` works; pass `rundle=` to disambiguate if a name happens to collide across rundles.

In [ ]:
PLAYER = tables.labels[tables.index_by_rundle[RUNDLE][0]][1]
print(f"Agreement between {PLAYER} and everyone in {RUNDLE}:")
for label, val in one_pairwise_agreement(tables, PLAYER, scope_rundle=RUNDLE):
    print(f"  {label[1]:<14} {val:.1%}")

## Whole-season queries

These scan across all ~36k rows instead of one rundle, so they're a lot heavier (multi-GB temporary arrays, tens of seconds). Run only if you have RAM headroom to spare.

In [ ]:
t0 = time.time()
season_avgs = average_agreement(tables)
print(f"computed in {time.time() - t0:.1f}s\n")
print("Top 10 average agreement (whole season):")
for label, avg in season_avgs[:10]:
    print(f"  {label[1]:<14} ({label[0]:<12}) {avg:.1%}" if avg is not None else f"  {label} n/a")

In [4]:
t0 = time.time()
top_pairs = most_similar_pairs(tables, n=10, min_non_forfeit=126)
print(f"computed in {time.time() - t0:.1f}s\n")
print("Top 10 most similar pairs (whole season):")
for (a, b), val in top_pairs:
    print(f"  {a[1]} ({a[0]}) & {b[1]} ({b[0]}): {val:.1%}")

computed in 35.6s

Top 10 most similar pairs (whole season):
  CalhounC (A_Patagonia) & CalhounJ3 (A_Patagonia): 98.7%
  VenguswamyK (A_Midland) & LloydP3 (A_Mojave): 96.8%
  SelzerE (A_Forest) & KightR (A_Polaris): 96.0%
  MunkD (A_Galaxy) & VenguswamyK (A_Midland): 95.8%
  JosephJ975 (A_Olive) & EilbacherP (B_Lighthouse): 95.7%
  MunkD (A_Galaxy) & LloydP3 (A_Mojave): 95.5%
  FriedmanJ4 (A_Bayou) & LloydP3 (A_Mojave): 95.5%
  ColwellB626 (A_Forest) & VenguswamyK (A_Midland): 95.5%
  BlairT (A_Cosmos) & HellendagI (A_Tranquility): 95.3%
  ColwellB626 (A_Forest) & LloydP3 (A_Mojave): 95.2%


In [5]:
one_pairwise_agreement(tables, "OzarowC", min_non_forfeit=126)

[(('C_Nebula_Div_2', 'MaD'), 0.8472222222222222),
 (('A_Seaboard', 'SheidlowerN'), 0.82),
 (('A_Waterfront', 'EbnerS'), 0.8055555555555556),
 (('A_Harbor', 'JonesJ2'), 0.8),
 (('C_Monolith_Div_1', 'ChangLucas'), 0.8),
 (('B_Badlands', 'BaramI'), 0.7986111111111112),
 (('A_Commonwealth', 'MaslykT'), 0.7933333333333333),
 (('A_Peninsula', 'LinJK'), 0.7933333333333333),
 (('B_Forest', 'FedorovD'), 0.7933333333333333),
 (('C_Outback_Div_2', 'DewanI'), 0.7916666666666666),
 (('B_Continental', 'ChandrashekarQ'), 0.7916666666666666),
 (('A_Zephyr', 'TolkinB'), 0.7916666666666666),
 (('B_Magnolia', 'YeungL'), 0.7866666666666666),
 (('A_Wilderness', 'CohenAa'), 0.7866666666666666),
 (('C_Atlantic_Div_2', 'LyonsJ'), 0.7866666666666666),
 (('A_Summit', 'WigdersonY'), 0.7866666666666666),
 (('C_Morningstar', 'BarlowE'), 0.7857142857142857),
 (('C_Volcano', 'RistowB'), 0.782608695652174),
 (('A_Nebula', 'SharpeR1'), 0.78),
 (('A_Boardwalk', 'KashyapA'), 0.78),
 (('C_Elysium_Div_2', 'ParkerP'), 0.78

In [6]:
one_pairwise_agreement(tables, "MaD", min_non_forfeit=126)

[(('A_Nebula', 'OzarowC'), 0.8472222222222222),
 (('A_Magnolia', 'EstyT'), 0.7986111111111112),
 (('C_Byzantium', 'NepplH'), 0.7916666666666666),
 (('D_Continental_Div_1', 'MandelbaumE2'), 0.7916666666666666),
 (('C_Meridian_Div_2', 'HegglandL'), 0.782608695652174),
 (('B_Orchard', 'LiK3'), 0.7777777777777778),
 (('B_Aspen', 'SuthersE'), 0.7777777777777778),
 (('D_Cypress_Div_2', 'MitraD2'), 0.7777777777777778),
 (('B_Meridian', 'SpryM'), 0.7753623188405797),
 (('A_Horizon', 'TaylorHJ'), 0.7708333333333334),
 (('C_Riviera_Div_2', 'Hawkins-PottierG'), 0.7708333333333334),
 (('B_Nebula', 'SimsC'), 0.7708333333333334),
 (('R_Div_27', 'DellaertT'), 0.7708333333333334),
 (('C_Morningstar', 'BarlowE'), 0.7698412698412699),
 (('C_Glacier_Div_2', 'MadgwickK'), 0.7651515151515151),
 (('D_Cove_Div_1', 'WangR4'), 0.7651515151515151),
 (('C_Lighthouse_Div_2', 'LeeA23'), 0.7651515151515151),
 (('B_Forest', 'FedorovD'), 0.7638888888888888),
 (('A_Cardinal', 'PorterDC'), 0.7638888888888888),
 (('C_Li

In [7]:
one_pairwise_agreement(tables, "ChapmanB2", min_non_forfeit=126)

[(('A_Continental', 'ThorlaksonG'), 0.8066666666666666),
 (('B_Boardwalk', 'MitchellM3'), 0.7933333333333333),
 (('A_Vista', 'LevineW'), 0.7866666666666666),
 (('A_Sahara', 'ThompsonC55'), 0.78),
 (('A_Yukon', 'LevineAS'), 0.7733333333333333),
 (('A_Nebula', 'RaoS91'), 0.7733333333333333),
 (('R_Div_71', 'SloanL'), 0.7733333333333333),
 (('B_Nebula', 'EhatammMattias'), 0.7708333333333334),
 (('A_Monolith', 'BorgiB'), 0.7681159420289855),
 (('C_Garden_Div_1', 'ArnesenS'), 0.7666666666666667),
 (('D_Nebula_Div_2', 'ColmanM'), 0.7666666666666667),
 (('B_Nebula', 'SimsC'), 0.7666666666666667),
 (('B_Atlantic', 'Kemeny S'), 0.7666666666666667),
 (('A_Olympic', 'ViswanathanKrishna'), 0.7666666666666667),
 (('C_Mesa_Div_2', 'GlennonS'), 0.76),
 (('D_Rainforest_Div_2', 'EpsteinB24'), 0.76),
 (('B_Highland', 'KolliJ'), 0.76),
 (('A_Valley', 'UzzellA'), 0.76),
 (('C_Atlantic_Div_2', 'LyonsJ'), 0.76),
 (('C_Maritime', 'BoreckiM'), 0.76),
 (('A_Maritime', 'AggarwalA'), 0.7575757575757576),
 (('B_M

In [5]:
average_agreement(tables)[:10]

/Users/ceruleanozarow/Downloads/LL_Analysis_V2/ll_all_rundle_analysis.py:376: RuntimeWarning: Mean of empty slice
  row_avg = np.nanmean(frac, axis=1)


[(('E_Delta_Div_1', 'WeichertM'), 0.6647493839263916),
 (('D_Rainforest_Div_2', 'WilderC2'), 0.657676637172699),
 (('E_Frontier_Div_2', 'GarabaduR'), 0.657676637172699),
 (('C_Memorial_Div_2', 'VincentEC'), 0.6386399865150452),
 (('E_Pampas_Div_2', 'HuberC2'), 0.6384621262550354),
 (('C_Oasis', 'WeissE2'), 0.637787401676178),
 (('D_Outback_Div_2', 'KoenigS2'), 0.6371365785598755),
 (('E_Sunrise_Div_2', 'RandallA212'), 0.6359637379646301),
 (('C_Junction_Div_1', 'HaberS2'), 0.6355755925178528),
 (('B_Arctic', 'AndersonE31'), 0.6348219513893127)]

In [6]:
h = average_agreement(tables, min_non_forfeit=126)

TypeError: average_agreement() got an unexpected keyword argument 'min_non_forfeit'